In [14]:
import numpy as np
import pandas as pd
import h5py
from scipy.io import loadmat
from scipy import stats
import os
from pathlib import Path
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [18]:
## functions for loading and sorting the calcium traces 
def normalize(trace, percentile=True):
    """ Normalize a fluorescence trace by its max or its 99th percentile. """
    trace = trace - np.min(trace)
    if np.percentile(trace, 99) > 0:
        if percentile:
            trace = trace / np.percentile(trace, 99)
        else:
            trace = trace / np.max(trace)
    return trace

def load_and_filter_traces(extract_mat_path: str,
                           labels_mat_path: str,
                           label_key: str = 'labels_ex'
                          ) -> pd.DataFrame:
    """
    Load temporal_weights from extract_mat_path (v7.3) and one of the
    labels_* arrays from labels_mat_path, keep only columns where
    labels == 1, and return as a pandas DataFrame.

    Parameters
    ----------
    extract_mat_path : str
        Path to the v7.3 .mat file containing an 'output/temporal_weights' dataset.
    labels_mat_path : str
        Path to the .mat file (v7.3 or earlier) containing a 'labels' group/struct.
    label_key : str, optional
        Which labels field to use: one of 'labels_ex', 'labels_ml', or
        'labels_overall'. Default is 'labels_ex'.

    Returns
    -------
    pd.DataFrame
        Rows = frames, columns = kept cells (named 'cell_<original_index>').
    """
    # validate choice
    allowed = ('labels_ex','labels_ml','labels_overall')
    if label_key not in allowed:
        raise ValueError(f"label_key must be one of {allowed}, got '{label_key}'")

    # 1) load & transpose temporal_weights → shape (nFrames, nCells)
    with h5py.File(extract_mat_path, 'r') as f:
        tw = f['output']['temporal_weights'][()]  # often (nCells, nFrames)
    tw = np.asarray(tw).T

    # 2) try to load chosen labels via HDF5; if that fails, fall back to loadmat
    try:
        with h5py.File(labels_mat_path, 'r') as f:
            hl = f['labels'][label_key][()]
    except OSError:
        mat = loadmat(labels_mat_path,
                      struct_as_record=False,
                      squeeze_me=True)
        lbl = mat['labels']  # either a dict or a mat_struct
        # pull out the right attribute/key
        if isinstance(lbl, dict):
            hl = lbl[label_key]
        else:
            hl = getattr(lbl, label_key)

    # 3) shape‐check & squeeze
    hl = np.asarray(hl).squeeze()
    if tw.shape[1] != hl.size:
        raise ValueError(
            f"dimension mismatch: temporal_weights is {tw.shape}, "
            f"labels array '{label_key}' has length {hl.size}"
        )

    # 4) filter & build DataFrame
    mask    = (hl == 1)
    kept_i  = np.nonzero(mask)[0]
    filtered = tw[:, mask]
    cols    = [f'cell_{i}' for i in kept_i]

    return (pd.DataFrame(filtered, columns=cols), hl)

def print_h5_tree(name, obj):
    """
    Callback for h5py.File.visititems.
    Prints group/dataset name and, for datasets, its shape and dtype.
    """
    if isinstance(obj, h5py.Group):
        print(f"Group:   {name}/")
    elif isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}  — shape={obj.shape}, dtype={obj.dtype}")

    
def zScoreTraces(dirPath, CNMFE_real_cells, miniscopeFramesPerSecond):
    
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    C_normalized = CNMFE_real_cells.apply(lambda col: normalize(col), axis=0)
    C_z_scored = CNMFE_real_cells.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(CNMFE_real_cells)-1)*(1/miniscopeFramesPerSecond), len(CNMFE_real_cells)), unit='s'), drop=True)
    C_normalized_z_scored = C_normalized.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_normalized)-1)*(1/miniscopeFramesPerSecond), len(C_normalized)), unit='s'), drop=True)

    save_path = dirPath / "_C_traces_filtered_origHz.csv"
    C_normalized_z_scored.to_csv(save_path)
    
    print('finished, saved:')
    print(save_path)
    
    return(C_normalized_z_scored)

def load_labels_struct(labels_mat_path):
    try:
        with h5py.File(labels_mat_path, 'r') as f:
            g = f['labels']
            out = {}
            for k in ['labels_ex', 'labels_ml', 'labels_overall']:
                out[k] = np.asarray(g[k][()]).squeeze().astype(np.int16)
            return out
    except OSError:
        mat = loadmat(labels_mat_path, struct_as_record=False, squeeze_me=True)
        lbl = mat['labels']
        out = {}
        for k in ['labels_ex', 'labels_ml', 'labels_overall']:
            out[k] = np.asarray(getattr(lbl, k)).squeeze().astype(np.int16)
        return out


def merge_human_labels(labels_ex_a, labels_ex_b):
    if labels_ex_a.shape != labels_ex_b.shape:
        raise ValueError(f"Shape mismatch: {labels_ex_a.shape} vs {labels_ex_b.shape}")

    merged = np.zeros_like(labels_ex_a, dtype=np.int16)
    merged[(labels_ex_a == -1) | (labels_ex_b == -1)] = -1
    merged[(merged == 0) & ((labels_ex_a == 1) | (labels_ex_b == 1))] = 1
    return merged


def recompute_overall(merged_labels_ex, labels_ml):
    if merged_labels_ex.shape != labels_ml.shape:
        raise ValueError(f"Shape mismatch: {merged_labels_ex.shape} vs {labels_ml.shape}")

    labels_overall = labels_ml.astype(np.int16).copy()
    human_labeled = merged_labels_ex != 0
    labels_overall[human_labeled] = merged_labels_ex[human_labeled]
    return labels_overall

In [3]:
#calcium analysis data - extract and ActSort labels 
dirPath = Path("/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/T_maze_data/ExtractOutput_052026/m326/20260603_224001_271")
GCaMPData_EXTRACT_fName = "_denoised_converted_concat_ds_mc_det_20260603_224001_271.mat"
ActSortLabels_fName = "precomputed_output_LABELS_JJM_withPredictions.mat"

extract_path = dirPath / GCaMPData_EXTRACT_fName
labels_path = dirPath / ActSortLabels_fName

print(extract_path.exists(), labels_path.exists())

True True


In [6]:
GCAMP_traces_humanSorted, labels_human = load_and_filter_traces(
    str(extract_path),
    str(labels_path),
    "labels_ex"
)

In [9]:
GCAMP_traces_ml, labels_ml = load_and_filter_traces(
    str(extract_path),
    str(labels_path),
    "labels_ml"
)

In [5]:
GCAMP_traces_Overall, labels_Overall = load_and_filter_traces(
    str(extract_path),
    str(labels_path),
    "labels_overall"
)

In [10]:
np.array_equal(labels_Overall, np.where(labels_human != 0, labels_human, labels_ml))

True

In [19]:
zScoreTraces(dirPath, GCAMP_traces_Overall, 20)

finished, saved:
/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/T_maze_data/ExtractOutput_052026/m326/20260603_224001_271/_C_traces_filtered_origHz.csv


,cell_0,cell_1,cell_8,cell_9,cell_16,cell_20,cell_21,cell_22,cell_23,cell_27,...,cell_207,cell_209,cell_217,cell_223,cell_225,cell_226,cell_235,cell_236,cell_237,cell_242
0 days 00:00:00,-0.248825,1.502958,-0.588964,0.464800,-0.573028,-0.614331,-0.193407,-0.414529,1.971959,0.972507,...,-0.427876,0.278360,-0.689758,-0.601340,-0.658930,-0.396751,-0.552790,0.024222,-0.455220,0.424612
0 days 00:00:00.050000,-0.524949,1.678598,0.059196,0.416933,-0.573028,-0.614331,-0.499145,-0.139476,0.805758,2.351655,...,-0.099547,0.351446,0.759733,-0.601340,-0.683473,0.388142,-0.499814,1.155138,-0.642758,-0.625483
0 days 00:00:00.100000,-0.524949,1.042079,-0.208998,0.111308,-0.315899,0.451482,-0.499145,-0.192734,1.175370,-0.670227,...,0.614135,-0.284902,0.210099,2.280584,-0.683473,-0.370805,0.070897,-0.087785,-0.642758,2.101267
0 days 00:00:00.150000,1.028117,-0.641561,-0.588964,-0.661641,-0.036285,-0.614331,-0.499145,-0.414529,0.080108,2.540379,...,-0.395263,-0.060911,-0.409594,-0.543737,-0.683473,-0.396751,0.099108,-0.222256,-0.256909,-0.625483
0 days 00:00:00.200000,0.259598,0.109391,0.178426,0.415520,-0.573028,1.383339,-0.499145,-0.414529,-0.269765,0.302079,...,-0.427876,-0.284902,0.986367,-0.601340,0.370688,-0.396751,-0.552790,0.111037,-0.642758,-0.244076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 02:35:59.200000,-0.274420,-0.107252,-0.588964,1.535896,0.618814,-0.084018,-0.499145,-0.414529,0.635754,1.182485,...,-0.427876,-0.284902,-0.689758,0.367048,-0.683473,1.177151,-0.552790,0.093251,0.038808,0.556507
0 days 02:35:59.250000,0.530036,-0.633779,-0.576832,-0.204388,-0.573028,-0.614331,1.026325,-0.414529,-0.546446,-0.181985,...,-0.323689,-0.284902,-0.689758,-0.601340,-0.683473,-0.037906,-0.339685,1.475697,2.845216,1.025488
0 days 02:35:59.300000,1.041190,-0.641561,-0.588964,0.931359,-0.238236,-0.155871,0.512099,-0.414529,-0.459415,0.435105,...,-0.427876,-0.284902,-0.262097,-0.518009,-0.683473,-0.396751,-0.107580,0.135285,1.653828,1.088589
0 days 02:35:59.350000,-0.207050,-0.161594,-0.588964,-0.177587,0.627909,0.907186,-0.499145,-0.414529,-0.546446,1.287382,...,-0.427876,-0.284902,0.753368,-0.601340,-0.683473,0.249991,-0.552790,-0.313061,0.401738,0.742309


In [12]:
save_path = dirPath / "_C_traces_filtered_origHz.csv"
GCAMP_traces_Overall.to_csv(save_path, index=False)
print(save_path)


/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/T_maze_data/ExtractOutput_052026/m326/20260603_224001_271/_C_traces_filtered_origHz.csv
